# Task
Build a Retrieval Augmented Generation (RAG) system using Pinecone for vector indexing and Groq's Llama model for text generation.

## Tải và Chuẩn bị Dữ liệu

### Subtask:
Tải dữ liệu của bạn vào môi trường và chuẩn bị nó (ví dụ: chia nhỏ thành các đoạn, làm sạch) để tạo embeddings.


Để bắt đầu, hãy chọn một tập dữ liệu mà bạn muốn sử dụng cho hệ thống RAG của mình. Dưới đây là một số tùy chọn:

1.  **Dữ liệu của riêng bạn**: Nếu bạn có các tài liệu văn bản, bài viết, hoặc tài liệu câu hỏi-trả lời riêng, bạn có thể tải chúng lên. Nếu dữ liệu của bạn nằm trong Google Drive, bạn có thể dễ dàng gắn kết nó vào Colab.
2.  **Tập dữ liệu công khai**: Bạn có thể sử dụng các tập dữ liệu công khai từ các thư viện như Hugging Face Datasets, Kaggle hoặc Wikipedia. Những tập dữ liệu này thường đã được cấu trúc tốt và sẵn sàng sử dụng.

### Hướng dẫn tải dữ liệu

Nếu bạn muốn tải dữ liệu từ Google Drive, hãy chạy đoạn mã sau để gắn kết Google Drive của bạn:

```python
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive đã được gắn kết.")
```

Sau khi Drive được gắn kết, bạn có thể truy cập các tệp của mình tại `/content/drive/My Drive/`. Ví dụ, nếu bạn có một tệp `my_data.csv` trong thư mục gốc của Drive, đường dẫn sẽ là `/content/drive/My Drive/my_data.csv`.

### Chuẩn bị dữ liệu để tạo nhúng (Embeddings)

Sau khi tải dữ liệu, bước tiếp theo là chuẩn bị dữ liệu đó để tạo embeddings. Các bước này thường bao gồm chia nhỏ văn bản thành các đoạn, làm sạch và chuẩn hóa.

Đầu tiên, chúng ta sẽ tải một tệp văn bản mẫu để minh họa các bước này. Nếu bạn đã có dữ liệu của mình, bạn có thể thay thế bước tải dữ liệu này bằng việc tải tệp của riêng bạn.

**Tải tệp văn bản mẫu (nếu bạn không có dữ liệu riêng):**

Bạn có thể sử dụng thư viện `requests` để tải một tệp văn bản từ một URL công khai. Ví dụ, chúng ta sẽ tải một bài viết mẫu từ Wikipedia.

**Reasoning**:
Following the explanation of data preparation, the next step is to provide the actual code to download a sample text file using the `requests` library, as indicated in the previous markdown block.



In [ ]:
import requests

# URL của một bài viết mẫu từ Wikipedia
url = "https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o"

# Tải nội dung văn bản
try:
    response = requests.get(url)
    response.raise_for_status() # Kiểm tra lỗi HTTP
    text_data = response.text
    print(f"Đã tải thành công dữ liệu từ {url}. Kích thước: {len(text_data)} ký tự.")

    # Lưu nội dung vào một tệp cục bộ
    with open("sample_article.txt", "w", encoding="utf-8") as f:
        f.write(text_data)
    print("Đã lưu nội dung vào 'sample_article.txt'")

except requests.exceptions.RequestException as e:
    print(f"Lỗi khi tải dữ liệu: {e}")

Lỗi khi tải dữ liệu: 403 Client Error: Forbidden for url: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o


## Gắn kết Google Drive và Trích xuất văn bản từ PDF

### Subtask:
Gắn kết Google Drive của bạn, sau đó duyệt qua thư mục 'ungthuda' trong 'My Drive' để tìm và trích xuất nội dung văn bản từ 3 tệp PDF của bạn.


**Reasoning**:
The first instruction is to mount Google Drive. I will provide the code to do that.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive đã được gắn kết.")

Mounted at /content/drive
Google Drive đã được gắn kết.


**Reasoning**:
The next step is to install the `pypdf` library, which is necessary for extracting text from PDF files, as specified in the subtask instructions.



In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 17.2 MB/s eta 0:00:00


**Reasoning**:
Now that Google Drive is mounted and `pypdf` is installed, I will define the path to the PDF directory, list the PDF files within it, and then extract text from each PDF as per the subtask instructions.



In [ ]:
import os
from pypdf import PdfReader

# Đường dẫn đến thư mục chứa các tệp PDF của bạn
pdf_folder_path = '/content/drive/My Drive/ungthuda/'

# Danh sách để lưu trữ nội dung văn bản đã trích xuất
extracted_texts = []

# Kiểm tra xem thư mục có tồn tại không
if not os.path.exists(pdf_folder_path):
    print(f"Lỗi: Thư mục '{pdf_folder_path}' không tồn tại. Vui lòng kiểm tra lại đường dẫn.")
else:
    # Lặp qua tất cả các tệp trong thư mục
    for filename in os.listdir(pdf_folder_path):
        if filename.endswith('.pdf'):
            filepath = os.path.join(pdf_folder_path, filename)
            print(f"Đang trích xuất từ: {filepath}")
            try:
                reader = PdfReader(filepath)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() + "\n"
                extracted_texts.append(text)
                print(f"Đã trích xuất thành công {len(reader.pages)} trang từ {filename}.")
            except Exception as e:
                print(f"Lỗi khi đọc tệp PDF {filename}: {e}")

print(f"Tổng cộng {len(extracted_texts)} tệp PDF đã được xử lý và trích xuất nội dung.")

# Bạn có thể xem một phần nội dung đã trích xuất từ tệp đầu tiên (nếu có)
if extracted_texts:
    print("\n--- Nội dung đã trích xuất từ tệp đầu tiên (một phần) ---")
    print(extracted_texts[0][:500]) # In 500 ký tự đầu tiên
else:
    print("Không tìm thấy tệp PDF nào để trích xuất hoặc có lỗi xảy ra.")

Lỗi: Thư mục '/content/drive/My Drive/ungthuda/' không tồn tại. Vui lòng kiểm tra lại đường dẫn.
Tổng cộng 0 tệp PDF đã được xử lý và trích xuất nội dung.
Không tìm thấy tệp PDF nào để trích xuất hoặc có lỗi xảy ra.


# Task
Để tiếp tục trích xuất văn bản từ các tệp PDF, chúng ta cần đảm bảo rằng đường dẫn đến thư mục chứa tệp PDF của bạn là chính xác.

Hiện tại, hệ thống báo lỗi rằng thư mục `'/content/drive/My Drive/ungthuda/'` không tồn tại.

Hãy kiểm tra lại đường dẫn này trên Google Drive của bạn. Bạn có thể làm theo các bước sau để xác nhận đường dẫn chính xác:

1.  Mở Google Drive của bạn trong trình duyệt web.
2.  Tìm đến thư mục `ungthuda` mà bạn muốn sử dụng.
3.  Click chuột phải vào thư mục `ungthuda` và chọn "Nhận đường liên kết" (Get link) hoặc tương tự. Điều này sẽ không cho bạn đường dẫn đầy đủ trong Colab, nhưng nó sẽ giúp bạn xác nhận tên và vị trí của thư mục.
4.  Trong Colab, bạn có thể chạy lệnh `!ls "/content/drive/My Drive/"` để liệt kê các thư mục và tệp trực tiếp trong "My Drive" và xác nhận tên thư mục `ungthuda` có tồn tại và được viết đúng chính tả hay không.

Sau khi xác nhận được đường dẫn chính xác, chúng ta sẽ cập nhật biến `pdf_folder_path` và chạy lại mã trích xuất.

Để giúp bạn kiểm tra, tôi sẽ cung cấp một đoạn mã để liệt kê nội dung của 'My Drive':

## Xác nhận Đường dẫn Google Drive

### Subtask:
Xác nhận đường dẫn chính xác đến thư mục chứa các tệp PDF trong Google Drive của bạn.


**Reasoning**:
The previous attempt to extract PDFs failed because the folder path was incorrect. To address this, the first step is to list the contents of 'My Drive' to help the user identify the correct folder path for their PDF files.



In [ ]:
print("Liệt kê nội dung của 'My Drive' để tìm thư mục PDF của bạn:")
!ls "/content/drive/My Drive/"


Liệt kê nội dung của 'My Drive' để tìm thư mục PDF của bạn:
ls: cannot access '/content/drive/My Drive/': No such file or directory


## Xác nhận Đường dẫn Google Drive

### Subtask:
Xác nhận đường dẫn chính xác đến thư mục chứa các tệp PDF trong Google Drive của bạn.


**Reasoning**:
The previous attempt to list 'My Drive' failed, indicating that Google Drive might not be properly mounted or accessible. Following the subtask's instructions, the first step is to re-mount Google Drive to ensure proper connection.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive đã được gắn kết.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive đã được gắn kết.


# Task
The Google Drive has been re-mounted. Now, let's list the contents of your 'My Drive' again. This will help you verify the correct path to your PDF folder, especially the 'ungthuda' directory.

```python
print("Liệt kê nội dung của 'My Drive' để tìm thư mục PDF của bạn:")
!ls "/content/drive/My Drive/"
```

## Cài đặt pypdf và Trích xuất Văn bản từ PDF

### Subtask:
Đảm bảo rằng thư viện `pypdf` đã được cài đặt và trích xuất nội dung văn bản từ các tệp PDF trong thư mục được chỉ định.


**Reasoning**:
I need to implement the detailed instructions for extracting text from PDF files, which involves updating the folder path, iterating through PDF files, and extracting their content page by page into a list, as well as providing status messages and a preview of the extracted text.



In [ ]:
import os
from pypdf import PdfReader

# 1. Cập nhật biến pdf_folder_path
pdf_folder_path = '/content/drive/My Drive/ungthuda/'

# 3. Danh sách để lưu trữ nội dung văn bản đã trích xuất
extracted_texts = []

# 4. Kiểm tra xem thư mục có tồn tại không
if not os.path.exists(pdf_folder_path):
    print(f"Lỗi: Thư mục '{pdf_folder_path}' không tồn tại. Vui lòng kiểm tra lại đường dẫn.")
else:
    print(f"Thư mục '{pdf_folder_path}' đã được tìm thấy. Bắt đầu trích xuất PDF...")
    # 5. Lặp qua tất cả các tệp trong thư mục
    for filename in os.listdir(pdf_folder_path):
        # 6. Đối với mỗi tệp có đuôi .pdf
        if filename.lower().endswith('.pdf'):
            filepath = os.path.join(pdf_folder_path, filename)
            print(f"Đang trích xuất từ: {filepath}")
            try:
                reader = PdfReader(filepath)
                text = ""
                for page_num, page in enumerate(reader.pages):
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                    else:
                        print(f"Cảnh báo: Trang {page_num + 1} của {filename} không có văn bản có thể trích xuất.")
                extracted_texts.append(text)
                print(f"Đã trích xuất thành công {len(reader.pages)} trang từ {filename}.")
            except Exception as e:
                print(f"Lỗi khi đọc tệp PDF {filename}: {e}")

# 7. In tổng số tệp PDF đã trích xuất văn bản thành công.
print(f"Tổng cộng {len(extracted_texts)} tệp PDF đã được xử lý và trích xuất nội dung.")

# 8. Để xác nhận, in 500 ký tự đầu tiên của văn bản được trích xuất đầu tiên (nếu có)
if extracted_texts:
    print("\n--- Nội dung đã trích xuất từ tệp đầu tiên (một phần) ---")
    print(extracted_texts[0][:500]) # In 500 ký tự đầu tiên
else:
    print("Không tìm thấy tệp PDF nào để trích xuất hoặc có lỗi xảy ra.")

Thư mục '/content/drive/My Drive/ungthuda/' đã được tìm thấy. Bắt đầu trích xuất PDF...
Đang trích xuất từ: /content/drive/My Drive/ungthuda/ClinicalDermatology.pdf
Cảnh báo: Trang 1 của ClinicalDermatology.pdf không có văn bản có thể trích xuất.
Đã trích xuất thành công 377 trang từ ClinicalDermatology.pdf.
Đang trích xuất từ: /content/drive/My Drive/ungthuda/Clinical Dermatology_ A Color Guide to Diagnosis and Therapy - PDF Room.pdf
Cảnh báo: Trang 1 của Clinical Dermatology_ A Color Guide to Diagnosis and Therapy - PDF Room.pdf không có văn bản có thể trích xuất.
Cảnh báo: Trang 12 của Clinical Dermatology_ A Color Guide to Diagnosis and Therapy - PDF Room.pdf không có văn bản có thể trích xuất.
Đã trích xuất thành công 1089 trang từ Clinical Dermatology_ A Color Guide to Diagnosis and Therapy - PDF Room.pdf.
Đang trích xuất từ: /content/drive/My Drive/ungthuda/understanding-skin-cancer-booklet.pdf
Đã trích xuất thành công 44 trang từ understanding-skin-cancer-booklet.pdf.
Đang tríc

## Chuẩn bị Dữ liệu cho Embeddings

### Subtask:
Chia nhỏ (chunk) văn bản đã trích xuất thành các đoạn có kích thước phù hợp và loại bỏ các đoạn rỗng hoặc quá ngắn.

**Reasoning**:
To prepare the extracted text for embeddings, we need to split it into smaller, overlapping chunks. This is a common practice in RAG systems to ensure that each chunk contains enough context for the embedding model while keeping the chunk size manageable. I will use `RecursiveCharacterTextSplitter` from `langchain_text_splitters` for this purpose.


In [ ]:
pip install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Khởi tạo bộ chia văn bản
# `chunk_size`: Kích thước tối đa của mỗi đoạn văn bản.
# `chunk_overlap`: Số lượng ký tự chồng lấn giữa các đoạn liên tiếp, giúp duy trì ngữ cảnh.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

# Danh sách để lưu trữ các đoạn văn bản đã được chia nhỏ
chunks = []

# Lặp qua nội dung văn bản đã trích xuất từ các tệp PDF
for i, text in enumerate(extracted_texts):
    if text.strip():  # Chỉ xử lý các văn bản không rỗng
        document_chunks = text_splitter.create_documents([text])
        # Thêm metadata để biết đoạn văn bản này đến từ tài liệu nào
        for j, chunk in enumerate(document_chunks):
            chunk.metadata = {"document_id": i, "chunk_id": j}
            chunks.append(chunk)
    else:
        print(f"Cảnh báo: Tài liệu số {i} không có văn bản hợp lệ để chia nhỏ.")

print(f"Tổng cộng {len(chunks)} đoạn văn bản đã được tạo.")

# In một vài đoạn đầu tiên để kiểm tra
if chunks:
    print("\n--- 5 đoạn văn bản đầu tiên (một phần) ---")
    for i, chunk in enumerate(chunks[:5]):
        print(f"Đoạn {i+1} (ID tài liệu: {chunk.metadata['document_id']}, ID đoạn: {chunk.metadata['chunk_id']}):")
        print(chunk.page_content[:200] + "...")  # In 200 ký tự đầu tiên của đoạn
        print("\n---\n")
else:
    print("Không có đoạn văn bản nào được tạo.")

Tổng cộng 11228 đoạn văn bản đã được tạo.

--- 5 đoạn văn bản đầu tiên (một phần) ---
Đoạn 1 (ID tài liệu: 0, ID đoạn: 0):
Clinical Dermatology
CD3A01  21/5/05  11:45 AM  Page i
For Ruth, Patricia and Arlene
CD3A01  21/5/05  11:45 AM  Page ii
Clinical Dermatology
J.A.A. Hunter
OBE BA MD FRCP (Edin)
Professor Emeritus of D...

---

Đoạn 2 (ID tài liệu: 0, ID đoạn: 1):
Blackwell Science Ltd, Osney Mead, Oxford OX2 0EL, UK
Blackwell Science Asia Pty Ltd, 550 Swanston Street, Carlton, Victoria 3053, Australia
The right of the Author to be identiﬁed as the Author of th...

---

Đoạn 3 (ID tài liệu: 0, ID đoạn: 2):
Hunter, J. A. A.
Clinical dermatology / J.A.A. Hunter, J.A. Savin, M.V. Dahl.— 3rd ed.
p. ; cm.
Includes index.
ISBN 0-632-05916-8
1. Skin—Diseases. 2. Dermatology.
[DNLM: 1. Skin Diseases—diagnosis. ...

---

Đoạn 4 (ID tài liệu: 0, ID đoạn: 3):
v
Contents
Preface to the third edition, vi
Preface to the ﬁrst edition, viii
Introduction, ix
1 Skin disease in perspective, 1
2 The fu

## Khởi tạo Pinecone

### Subtask:
Cấu hình API key và environment của Pinecone mà bạn đã cung cấp, sau đó khởi tạo kết nối. Code sẽ kiểm tra và tạo một chỉ mục mới nếu nó chưa tồn tại, để lưu trữ các vector embeddings.

**Reasoning**:
Before we can interact with Pinecone, we need to install its client library. This step ensures that all necessary dependencies are met.


In [ ]:
pip install -U pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


**Reasoning**:
Now that the Pinecone client is installed, we need to initialize Pinecone using your API key and environment. The code will then check if the specified index already exists. If not, it will create a new index with the desired dimension and metric type. This prepares Pinecone to store the embeddings.

Để tiếp tục, bạn cần cung cấp `PINECONE_API_KEY` và `PINECONE_ENVIRONMENT` của mình. Vui lòng thêm các khóa này vào Secret Manager của Colab (biểu tượng khóa ở bên trái) với tên `PINECONE_API_KEY` và `PINECONE_ENVIRONMENT`.

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

# CẢNH BÁO: KHÔNG ĐƯỢC LÀM ĐIỀU NÀY TRONG MÔI TRƯỜNG SẢN XUẤT THỰC TẾ!
# Để giải quyết vấn đề tạm thời, bạn có thể dán API Key và Environment trực tiếp vào đây.
# Vui lòng dán chính xác API Key và Environment của bạn vào hai biến dưới đây:
PINECONE_API_KEY = ''  # <-- DÁN API KEY CỦA BẠN VÀO ĐÂY
PINECONE_ENVIRONMENT = 'us-east-1' # <-- DÁN ENVIRONMENT CỦA BẠN VÀO ĐÂY (ví dụ: 'us-east-1', 'gcp-starter', 'asia-southeast1')

# Đảm bảo loại bỏ ký tự trắng thừa nếu có
PINECONE_API_KEY = PINECONE_API_KEY.strip()
PINECONE_ENVIRONMENT = PINECONE_ENVIRONMENT.strip()

# Khởi tạo Pinecone
pinecone = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENVIRONMENT)

index_name = 'ungthuda-rag-index2'  # Đặt tên cho index của bạn
dimension = 768  # Kích thước embedding sẽ được tạo bởi mô hình, sẽ được xác định sau
metric = 'cosine'  # Metric để đo độ tương đồng giữa các vector

# Kiểm tra và tạo index nếu chưa tồn tại
if index_name not in pinecone.list_indexes():
    print(f"Đang tạo index '{index_name}'...")
    pinecone.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=ServerlessSpec(
            cloud='aws',
            region=PINECONE_ENVIRONMENT # Sử dụng PINECONE_ENVIRONMENT làm region
        )
    )
    print(f"Index '{index_name}' đã được tạo thành công.")
else:
    print(f"Index '{index_name}' đã tồn tại.")

# Kết nối đến index
index = pinecone.Index(index_name)
print(f"Đã kết nối đến index '{index_name}'.")
print("Trạng thái index:")
print(index.describe_index_stats())

Đang tạo index 'ungthuda-rag-index2'...
Index 'ungthuda-rag-index2' đã được tạo thành công.
Đã kết nối đến index 'ungthuda-rag-index2'.
Trạng thái index:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 10 Mar 2026 00:48:24 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '35',
                                    'x-pinecone-request-latency-ms': '34',
                                    'x-pinecone-response-duration-ms': '37'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


## Nhúng và Lập chỉ mục Dữ liệu vào Pinecone

### Subtask:
Sử dụng một mô hình nhúng (embedding model) để chuyển đổi các đoạn văn bản đã chuẩn bị thành các vector embeddings. Các vector này sau đó sẽ được chèn vào chỉ mục Pinecone cùng với metadata tương ứng.

**Reasoning**:
To create embeddings for our text chunks, we need an embedding model. We'll use a `SentenceTransformer` model which is suitable for this task and can generate 768-dimensional embeddings, matching our Pinecone index configuration. We also need to install the `sentence-transformers` library first.

In [ ]:
pip install -U sentence-transformers

**Reasoning**:
Now that `sentence-transformers` is installed, we can load an appropriate model and generate embeddings for each text chunk. We will then prepare these embeddings and their associated metadata for upserting into our Pinecone index.

In [ ]:
from sentence_transformers import SentenceTransformer
import time

# Khởi tạo mô hình embedding
# 'all-MiniLM-L6-v2' là một mô hình phổ biến, tạo ra embeddings 384 chiều.
# Tuy nhiên, index Pinecone của chúng ta được tạo với dimension 768.
# Chúng ta sẽ sử dụng một mô hình khác tạo ra embeddings 768 chiều.
# Ví dụ: 'BAAI/bge-small-en-v1.5' (384 chiều) hoặc 'sentence-transformers/all-mpnet-base-v2' (768 chiều)
# Cảnh báo: Mô hình 768 chiều sẽ tốn nhiều tài nguyên hơn.
embedding_model_name = 'sentence-transformers/all-mpnet-base-v2'
print(f"Đang tải mô hình embedding: {embedding_model_name}...")
embedding_model = SentenceTransformer(embedding_model_name)
print("Mô hình embedding đã được tải.")

# Tạo embeddings và chuẩn bị dữ liệu để upsert vào Pinecone
# Định dạng dữ liệu: [(id, vector, metadata), ...]
upsert_data = []
batch_size = 100  # Upsert theo từng batch để tránh lỗi timeout

print(f"Bắt đầu tạo embeddings và chuẩn bị dữ liệu cho {len(chunks)} đoạn văn bản...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    batch_texts = [chunk.page_content for chunk in batch_chunks]

    # Tạo embeddings cho batch hiện tại
    batch_embeddings = embedding_model.encode(batch_texts, show_progress_bar=False)

    # Chuẩn bị dữ liệu cho upsert
    for j, chunk in enumerate(batch_chunks):
        # Tạo ID duy nhất cho mỗi chunk
        unique_id = f"doc_{chunk.metadata['document_id']}_chunk_{chunk.metadata['chunk_id']}"
        metadata = {"text": chunk.page_content, **chunk.metadata}
        upsert_data.append((unique_id, batch_embeddings[j].tolist(), metadata))

    print(f"Đã xử lý {len(upsert_data)}/{len(chunks)} đoạn văn bản.")

print("Đã hoàn tất việc tạo embeddings. Bắt đầu upsert vào Pinecone...")

# Upsert dữ liệu vào Pinecone
try:
    # Pinecone upsert cũng hỗ trợ batching nội bộ, nhưng chúng ta đã tự batch rồi.
    # Lệnh upsert sẽ tự động chia nhỏ nếu vượt quá giới hạn request.
    index.upsert(vectors=upsert_data, batch_size=batch_size)
    print(f"Đã upsert thành công {len(upsert_data)} vector vào index '{index_name}'.")
except Exception as e:
    print(f"Lỗi khi upsert dữ liệu vào Pinecone: {e}")

# In trạng thái của index sau khi upsert
print("Trạng thái index sau khi upsert:")
print(index.describe_index_stats())


Đang tải mô hình embedding: sentence-transformers/all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mô hình embedding đã được tải.
Bắt đầu tạo embeddings và chuẩn bị dữ liệu cho 11228 đoạn văn bản...
Đã xử lý 100/11228 đoạn văn bản.
Đã xử lý 200/11228 đoạn văn bản.
Đã xử lý 300/11228 đoạn văn bản.
Đã xử lý 400/11228 đoạn văn bản.
Đã xử lý 500/11228 đoạn văn bản.
Đã xử lý 600/11228 đoạn văn bản.
Đã xử lý 700/11228 đoạn văn bản.
Đã xử lý 800/11228 đoạn văn bản.
Đã xử lý 900/11228 đoạn văn bản.
Đã xử lý 1000/11228 đoạn văn bản.
Đã xử lý 1100/11228 đoạn văn bản.
Đã xử lý 1200/11228 đoạn văn bản.
Đã xử lý 1300/11228 đoạn văn bản.
Đã xử lý 1400/11228 đoạn văn bản.
Đã xử lý 1500/11228 đoạn văn bản.
Đã xử lý 1600/11228 đoạn văn bản.
Đã xử lý 1700/11228 đoạn văn bản.
Đã xử lý 1800/11228 đoạn văn bản.
Đã xử lý 1900/11228 đoạn văn bản.
Đã xử lý 2000/11228 đoạn văn bản.
Đã xử lý 2100/11228 đoạn văn bản.
Đã xử lý 2200/11228 đoạn văn bản.
Đã xử lý 2300/11228 đoạn văn bản.
Đã xử lý 2400/11228 đoạn văn bản.
Đã xử lý 2500/11228 đoạn văn bản.
Đã xử lý 2600/11228 đoạn văn bản.
Đã xử lý 2700/11228 đoạn 

Upserted vectors:   0%|          | 0/11228 [00:00<?, ?it/s]

Đã upsert thành công 11228 vector vào index 'ungthuda-rag-index2'.
Trạng thái index sau khi upsert:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 10 Mar 2026 00:54:22 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '42',
                                    'x-pinecone-request-latency-ms': '41',
                                    'x-pinecone-response-duration-ms': '43'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 11228}},
 'storageFullness': 0.0,
 'total_vector_count': 11228,
 'vector_type': 'dense'}


## Cấu hình Mô hình Llama của Groq

### Subtask:
Thiết lập API key của Groq và chọn mô hình Llama bạn muốn sử dụng để tạo văn bản.

**Reasoning**:
Before we can use Groq's Llama model, we need to install the `groq` Python client library. This ensures all necessary dependencies are in place to interact with the Groq API.

In [ ]:
pip install -U groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 12.6 MB/s eta 0:00:00


**Reasoning**:
Now that the `groq` client library is installed, we need to initialize the Groq client using your API key. This will allow us to interact with their LLM services, including the Llama models. You will need to provide your `GROQ_API_KEY`.

Để tiếp tục, bạn cần cung cấp `GROQ_API_KEY` của mình. Vui lòng thêm khóa này vào Secret Manager của Colab (biểu tượng khóa ở bên trái) với tên `GROQ_API_KEY`.

In [ ]:
from groq import Groq
from google.colab import userdata

# CẢNH BÁO: KHÔNG ĐƯỢC LÀM ĐIỀU NÀY TRONG MÔI TRƯỜNG SẢN XUẤT THỰC TẾ!
# Để giải quyết vấn đề tạm thời, bạn có thể dán API Key trực tiếp vào đây.
# Vui lòng dán chính xác API Key của bạn vào biến dưới đây:
GROQ_API_KEY = '' # <-- DÁN GROQ API KEY CỦA BẠN VÀO ĐÂY

# Khởi tạo Groq client
client = Groq(api_key=GROQ_API_KEY)

print("Groq client đã được khởi tạo.")

# Tên mô hình Llama bạn muốn sử dụng
# Ví dụ: 'llama-3.1-8b-instant', 'llama-3.1-70b-versatile', 'meta-llama/llama-4-maverick-17b-128e-instruct'
# Bạn có thể kiểm tra các mô hình có sẵn trên trang chủ của Groq hoặc qua API của họ.
GROQ_MODEL_NAME = 'meta-llama/llama-4-maverick-17b-128e-instruct'
print(f"Mô hình Llama đã chọn: {GROQ_MODEL_NAME}")

Groq client đã được khởi tạo.
Mô hình Llama đã chọn: meta-llama/llama-4-maverick-17b-128e-instruct


## Triển khai RAG Query và Tạo Văn bản

### Subtask:
Phát triển một chức năng để nhận truy vấn từ người dùng, tìm kiếm các tài liệu liên quan trong Pinecone, sau đó sử dụng các tài liệu này và truy vấn để tạo ra câu trả lời bằng mô hình Llama của Groq.

**Reasoning**:
Now that our Pinecone index is populated with embeddings and the Groq client is configured, we can create the core RAG function. This function will take a user query, embed it, search Pinecone for relevant documents, and then use those documents as context for the Groq Llama model to generate a response.

In [ ]:
def rag_query(query: str, top_k: int = 3) -> str:
    """
    Thực hiện truy vấn RAG: tạo embedding cho truy vấn, tìm kiếm trong Pinecone,
    và tạo câu trả lời bằng mô hình Llama của Groq.

    Args:
        query (str): Câu hỏi của người dùng.
        top_k (int): Số lượng tài liệu liên quan nhất sẽ được truy xuất từ Pinecone.

    Returns:
        str: Câu trả lời được tạo bởi mô hình Llama của Groq.
    """
    print(f"Truy vấn: {query}")

    # 1. Tạo embedding cho truy vấn
    query_embedding = embedding_model.encode(query).tolist()

    # 2. Tìm kiếm các tài liệu liên quan nhất trong Pinecone
    print(f"Đang tìm kiếm {top_k} tài liệu liên quan trong Pinecone...")
    search_results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # 3. Thu thập các đoạn văn bản (context) từ kết quả tìm kiếm
    contexts = []
    for match in search_results.matches:
        contexts.append(match.metadata['text'])

    if not contexts:
        print("Không tìm thấy tài liệu liên quan trong Pinecone. Trả lời dựa trên kiến thức chung.")
        context_string = ""
    else:
        context_string = "\n\n".join(contexts)
        print(f"Đã truy xuất {len(contexts)} đoạn văn bản làm ngữ cảnh.")

    # 4. Chuẩn bị prompt cho mô hình Groq Llama
    if context_string:
        prompt_messages = [
            {
                "role": "system",
                "content": "Bạn là một trợ lý hữu ích. Hãy sử dụng thông tin ngữ cảnh được cung cấp để trả lời câu hỏi. Nếu thông tin không có trong ngữ cảnh, hãy nói rằng bạn không biết, đừng cố bịa ra câu trả lời."
            },
            {
                "role": "user",
                "content": f"Ngữ cảnh: {context_string}\n\nCâu hỏi: {query}"
            }
        ]
    else:
         prompt_messages = [
            {
                "role": "system",
                "content": "Bạn là một trợ lý hữu ích. Hãy trả lời câu hỏi dựa trên kiến thức của bạn. Nếu bạn không biết, hãy nói rằng bạn không biết."
            },
            {
                "role": "user",
                "content": f"Câu hỏi: {query}"
            }
        ]

    # 5. Gọi mô hình Groq Llama để tạo câu trả lời
    print("Đang tạo câu trả lời bằng mô hình Groq Llama...")
    chat_completion = client.chat.completions.create(
        messages=prompt_messages,
        model=GROQ_MODEL_NAME,
        temperature=0.7, # Điều chỉnh nhiệt độ để kiểm soát tính sáng tạo của câu trả lời
        max_tokens=1024 # Giới hạn số lượng token trong câu trả lời
    )

    response_content = chat_completion.choices[0].message.content
    print("Đã hoàn thành tạo câu trả lời.")
    return response_content

# Ví dụ sử dụng chức năng RAG
print("\n--- THỬ NGHIỆM RAG QUERY ---")
user_query = "Da liễu là gì và nó bao gồm những bệnh gì?"
answer = rag_query(user_query)
print(f"\nCâu trả lời: {answer}")

user_query_2 = "Kể tên các loại thuốc bôi ngoài da phổ biến?"
answer_2 = rag_query(user_query_2)
print(f"\nCâu trả lời: {answer_2}")

user_query_3 = "Làm thế nào để xây dựng một tên lửa?" # Một câu hỏi không có trong ngữ cảnh
answer_3 = rag_query(user_query_3)
print(f"\nCâu trả lời: {answer_3}")


--- THỬ NGHIỆM RAG QUERY ---
Truy vấn: Da liễu là gì và nó bao gồm những bệnh gì?
Đang tìm kiếm 3 tài liệu liên quan trong Pinecone...
Đã truy xuất 3 đoạn văn bản làm ngữ cảnh.
Đang tạo câu trả lời bằng mô hình Groq Llama...
Đã hoàn thành tạo câu trả lời.

Câu trả lời: Da liễu là một chuyên ngành y khoa chuyên về chẩn đoán, điều trị và phòng ngừa các bệnh liên quan đến da, tóc, móng và các vấn đề khác liên quan đến da. Da liễu bao gồm nhiều bệnh khác nhau, từ các bệnh thông thường như mụn trứng cá, viêm da tiếp xúc, nấm da, đến các bệnh nghiêm trọng hơn như ung thư da, bệnh tự miễn như vảy nến, lupus ban đỏ.

Trong ngữ cảnh y khoa, da liễu cũng liên quan đến các bệnh lây truyền qua đường tình dục (STDs) như trong tài liệu được cung cấp, đề cập đến các bệnh như chancroid và lymphogranuloma venereum (LGV), cả hai đều có thể gây ra các triệu chứng ở vùng sinh dục và có thể được điều trị bằng kháng sinh.

Da liễu cũng bao gồm cả các bệnh lý về thẩm mỹ như sẹo, rụng tóc, và các vấn đề về s